In [2]:
import numpy as np
from sklearn import preprocessing

# Blok 1: Załadowanie pliku CSV i wydzielenie wejść oraz targetów
raw_csv_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')
unscaled_inputs_all = raw_csv_data[:, 1:-1]
targets_all = raw_csv_data[:, -1]


In [3]:
# Blok 2: Przetasowanie danych przed balansowaniem
shuffled_indices = np.arange(unscaled_inputs_all.shape[0])
np.random.shuffle(shuffled_indices)
unscaled_inputs_all = unscaled_inputs_all[shuffled_indices]
targets_all = targets_all[shuffled_indices]

In [4]:
# Blok 3: Zrównoważenie zbioru danych (usunięcie nadmiaru zer)
num_one_targets = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove = []

In [5]:
# Blok 4: Normalizacja danych wejściowych
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

In [6]:
# Blok 5: Ponowne przetasowanie znormalizowanych danych
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)
shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equal_priors[shuffled_indices]

In [7]:
# Blok 6: Podział na zbiór treningowy, walidacyjny i testowy
samples_count = shuffled_inputs.shape[0]

train_samples_count = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)
test_samples_count = samples_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[:train_samples_count]
train_targets = shuffled_targets[:train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets = shuffled_targets[train_samples_count+validation_samples_count:]

In [8]:
# Blok 7: Sprawdzenie balansu i zapis do plików .npz
print(np.sum(train_targets), train_samples_count, np.sum(train_targets) / train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets) / validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets) / test_samples_count)

np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)

1782.0 3579 0.4979044425817267
223.0 447 0.4988814317673378
232.0 448 0.5178571428571429


In [13]:
import numpy as np
import tensorflow as tf
npz = np.load('Audiobooks_data_train.npz')
train_inputs = npz['inputs'].astype(np.float32)
train_targets = npz['targets'].astype(np.int32)
npz = np.load('Audiobooks_data_validation.npz')
validation_inputs, validation_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_test.npz')
test_inputs, test_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

In [14]:
input_size = 10
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),      # Pierwsza warstwa ukryta
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),      # Druga warstwa ukryta
    tf.keras.layers.Dense(output_size, activation='softmax')          # Warstwa wyjściowa
])

In [15]:
input_size = 10
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),      # Pierwsza warstwa ukryta
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),      # Druga warstwa ukryta
    tf.keras.layers.Dense(output_size, activation='softmax')          # Warstwa wyjściowa
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [16]:
batch_size = 100
max_epochs = 100
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)

model.fit(train_inputs,
          train_targets,
          batch_size=batch_size,
          epochs=max_epochs,
          callbacks=[early_stopping],
          validation_data=(validation_inputs, validation_targets),
          verbose=1
          )

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6186 - loss: 0.6285 - val_accuracy: 0.7248 - val_loss: 0.5344
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7494 - loss: 0.5128 - val_accuracy: 0.7629 - val_loss: 0.4653
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7664 - loss: 0.4573 - val_accuracy: 0.7830 - val_loss: 0.4311
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7793 - loss: 0.4290 - val_accuracy: 0.7875 - val_loss: 0.4170
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7896 - loss: 0.4120 - val_accuracy: 0.7919 - val_loss: 0.4018
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7988 - loss: 0.3994 - val_accuracy: 0.7919 - val_loss: 0.3934
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7974 - loss: 0.3916 - val_accuracy: 0.8009 - val_loss: 0.3845
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7991 - loss: 0.3853 - val_accuracy: 0.8098 - v

In [17]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print('\nTest loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy * 100.))

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7879 - loss: 0.4019

Test loss: 0.40. Test accuracy: 78.79%
